### MACS 30123 Lab Session: Working with EMR Clusters/Spark
*Week 7*

**Edited by Yangyu Wang (developed from Ethan Kozlowski, Nalin Bhatt, Max Zhu, Adam Wu, Wonje Yun)**

### Create S3 Bucket

First create S3 bucket to store our files.

In [6]:
import boto3

In [ ]:
# Initialize boto3 handler
s3 = boto3.resource('s3')

# Create a new bucket to store your files
BUCKETNAME = 'example-bucket'
s3.create_bucket(Bucket=BUCKETNAME)

# This is what we will use to interface with the specific bucket
bucket = s3.Bucket( BUCKETNAME )

In [ ]:
# Upload your files to S3

FILENAME = '../exercise/lab_wk7_spark.py'
with open(FILENAME, 'rb') as myfile:
    bucket.put_object(Key='lab_wk7_spark.py', Body=myfile)

FILENAME = '../exercise/lab_wk7_sparksql.ipynb'
with open(FILENAME, 'rb') as myfile:
    bucket.put_object(Key='lab_wk7_sparksql.ipynb', Body=myfile)

### Launching EMR Cluster

Next launch EMR Cluster in Terminal/bash.

In [ ]:
%%bash 

python launch_spark_cluster.py \
    --s3_bucket YOUR_BUCKET_NAME \
    --primary_count 1 \
    --core_count 2 \
    --instance_type "m5.xlarge"

{
    "ClusterId": "j-2GGSSO7FCKI2X",
    "ClusterArn": "arn:aws:elasticmapreduce:us-east-1:348752177325:cluster/j-2GGSSO7FCKI2X"
}


When creating a new cluster, make sure to adjust the security settings to allow for `ssh` access. See `emr_cheatsheet.md` in Week 7 course materials.

#### Method 1: `ssh` Directly

The first way to work with EMR is to directly `ssh` into it, then work with it just like we did for `EC2` (see previous lab on EC2).

Connecting to it:
```
$ ssh -i vockey.pem hadoop@EMR-PUBLIC-ADDRESS
```

Uploading a folder called `mystuff` locally -> EMR:
```
$ scp -i "vockey.pem" -r mystuff hadoop@EMR-PUBLIC-ADDRESS:/home/hadoop
```

Downloading a folder called `mystuff` from EMR -> locally:
```
$ scp -i "vockey.pem" -r hadoop@EMR-PUBLIC-ADDRESS:/home/hadoop/mystuff .
```
---

After uploading your files in there, you can then run Spark jobs with
``` 
[EMR] spark-submit mystuff/myfile.py
```
Alternatively if your files are saved on `S3`, then
```
[EMR] spark-submit s3://example-bucket/mystuff/myfile.py example-bucket
```

#### Method 2: Interactive Sessions

You can also launch a Jupyter server directly on EMR and work with it interactively.
```
$ ssh -i "vockey.pem" -NL 9443:localhost:9443 hadoop@EMR-PUBLIC-ADDRESS
```
This forwards the remote connection to your `https://localhost:9443`, and you can log in with username `jovyan`, password `jupyter`. 

### Terminating EMR Cluster

In [ ]:
!python terminate_spark_cluster.py --cluster_id j-XXXXXXXXXXXXX